In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor, BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor, StackingRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import mlflow
from mlflow.models import infer_signature
from xgboost import XGBRegressor

In [2]:
fr = pd.read_csv('Cleaned_data_version_2.csv', index_col= None)

fr = pd.DataFrame(fr)

fr.drop('Unnamed: 0', axis=1, inplace = True)

fr

,job_title_AI Engineer,job_title_Analytics Engineer,job_title_Business Intelligence Analyst,job_title_Computer Vision Engineer,job_title_Data Analyst,job_title_Data Engineer,job_title_Data Science Manager,job_title_Data Scientist,job_title_LLM Engineer,job_title_MLOps Engineer,...,industry_Healthcare,industry_Manufacturing,industry_Media,industry_Retail,industry_Technology,employee_residence_IN,education_level,company_size,years_experience,salary_usd
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0,1.252729,23844
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,3.0,2,-0.157964,170132
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,-1.734621,83059
3,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1,0.277691,29531
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1,-1.070766,70105
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1,0.194709,38810
4996,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1,1.024528,123368
4997,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,2.0,1,-0.054237,86123
4998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,2.0,1,0.941546,76001


In [3]:
X = fr.drop(columns=["salary_usd"] , axis=1)
y = fr["salary_usd"]

In [5]:
#Data Splitting

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [6]:
#Cross Validation

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [7]:
#Models


models = {

    "Linear Regression": {
        "model": LinearRegression(),
        "params": {}
    },

    "Ridge": {
        "model": Ridge(),
        "params": {
            "alpha": [0.01, 0.1, 1, 10, 100, 1000]
        }
    },

    "Lasso": {
        "model": Lasso(max_iter=20000),
        "params": {
            "alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10]
        }
    },

    "ElasticNet": {
        "model": ElasticNet(max_iter=20000),
        "params": {
            "alpha": [0.0001, 0.001, 0.01, 0.1, 1],
            "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
        }
    },

    "Decision Tree": {
        "model": DecisionTreeRegressor(random_state=42),
        "params": {
            "max_depth": [None, 5, 10, 15, 20, 30],
            "min_samples_split": [2, 5, 10, 20],
            "min_samples_leaf": [1, 2, 5, 10],
            "max_features": [None, "sqrt", "log2"]
        }
    },

    "Random Forest": {
        "model": RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        ),
        "params": {
            "n_estimators": [200, 500],
            "max_depth": [None, 10, 20, 30],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
            "max_features": [1.0, "sqrt", "log2"]
        }
    },

    "Extra Trees": {
        "model": ExtraTreesRegressor(
            random_state=42,
            n_jobs=-1
        ),
        "params": {
            "n_estimators": [200, 500],
            "max_depth": [None, 10, 20, 30],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
            "max_features": [1.0, "sqrt", "log2"]
        }
    },

    "Gradient Boosting": {
        "model": GradientBoostingRegressor(
            random_state=42
        ),
        "params": {
            "n_estimators": [100, 200, 300],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [2, 3, 5],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 5],
            "subsample": [0.8, 1.0]
        }
    },

    "Hist Gradient Boosting": {
        "model": HistGradientBoostingRegressor(
            random_state=42
        ),
        "params": {
            "learning_rate": [0.03, 0.05, 0.1],
            "max_iter": [100, 200, 300],
            "max_leaf_nodes": [15, 31, 63],
            "max_depth": [None, 5, 10],
            "min_samples_leaf": [10, 20, 30],
            "l2_regularization": [0, 0.1, 1]
        }
    },

    "KNN": {
        "model": KNeighborsRegressor(),
        "params": {
            "n_neighbors": [3, 5, 7, 10, 15, 20],
            "weights": ["uniform", "distance"],
            "p": [1, 2]
        }
    },

    "SVR": {
        "model": SVR(),
        "params": {
            "kernel": ["rbf", "linear"],
            "C": [0.1, 1, 10, 100],
            "epsilon": [0.01, 0.1, 0.2, 0.5],
            "gamma": ["scale", "auto"]
        }
    },

    "XGBoost": {
        "model": XGBRegressor(
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        ),
        "params": {
            "n_estimators": [200, 500],
            "learning_rate": [0.03, 0.05, 0.1],
            "max_depth": [3, 5, 7],
            "min_child_weight": [1, 3],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0],
            "reg_alpha": [0, 0.1],
            "reg_lambda": [1, 5]
        }
    }
}

In [8]:
results = []

best_models = {}

for name, config in models.items():

    print("=" * 70)
    print(f"Training: {name}")

    grid = GridSearchCV(
        estimator=config["model"],
        param_grid=config["params"],
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)

    # Best model found by GridSearch
    best_model = grid.best_estimator_

    # Store best model
    best_models[name] = best_model

    # Predict on test set
    y_pred = best_model.predict(X_test)

    # Evaluation metrics
    r2 = r2_score(y_test, y_pred)

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(y_test, y_pred)
    )

    # Store results
    results.append({
        "Model": name,
        "CV RMSE": -grid.best_score_,
        "Test R2": r2,
        "Test MAE": mae,
        "Test RMSE": rmse,
        "Best Parameters": grid.best_params_
    })

    print(f"Best CV RMSE : {-grid.best_score_:,.2f}")
    print(f"Test R²      : {r2:.4f}")
    print(f"Test MAE     : {mae:,.2f}")
    print(f"Test RMSE    : {rmse:,.2f}")
    print(f"Best Params  : {grid.best_params_}")

Training: Linear Regression
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Best CV RMSE : 23,918.99
Test R²      : 0.8018
Test MAE     : 18,333.10
Test RMSE    : 24,554.69
Best Params  : {}
Training: Ridge
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best CV RMSE : 23,918.45
Test R²      : 0.8017
Test MAE     : 18,332.71
Test RMSE    : 24,560.53
Best Params  : {'alpha': 1}
Training: Lasso
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best CV RMSE : 23,918.96
Test R²      : 0.8018
Test MAE     : 18,332.48
Test RMSE    : 24,555.68
Best Params  : {'alpha': 1}
Training: ElasticNet
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best CV RMSE : 23,918.43
Test R²      : 0.8017
Test MAE     : 18,332.71
Test RMSE    : 24,561.85
Best Params  : {'alpha': 0.001, 'l1_ratio': 0.7}
Training: Decision Tree
Fitting 5 folds for each of 288 candidates, totalling 1440 fits
Best CV RMSE : 27,988.75
Test R²      : 0.7218
Test MAE     : 20,304.64
Test RM

In [9]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Test R2",
    ascending=False
)

results_df



comparison = results_df[
    [
        "Model",
        "CV RMSE",
        "Test R2",
        "Test MAE",
        "Test RMSE"
    ]
]

comparison

,Model,CV RMSE,Test R2,Test MAE,Test RMSE
11,XGBoost,20874.332422,0.849473,15077.013672,21400.835872
8,Hist Gradient Boosting,20760.235511,0.848331,15088.300360,21481.864901
7,Gradient Boosting,21228.144841,0.846643,15466.277053,21601.047013
6,Extra Trees,22727.977349,0.815196,16952.527082,23712.567980
5,Random Forest,23230.539213,0.805609,17457.814043,24319.854612
0,Linear Regression,23918.986063,0.801837,18333.101466,24554.688418
2,Lasso,23918.956434,0.801821,18332.480990,24555.682845
1,Ridge,23918.453920,0.801743,18332.713277,24560.526929
3,ElasticNet,23918.432251,0.801722,18332.714796,24561.853918
9,KNN,28569.861382,0.726066,21584.571339,28869.953715


In [10]:
best_model_name = results_df.iloc[0]["Model"]

best_model = best_models[best_model_name]

print("Best Model:", best_model_name)
print("\nBest Model Parameters:")
print(best_model)

Best Model: XGBoost

Best Model Parameters:
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=3, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=-1, num_parallel_tree=None, ...)


In [11]:
import joblib

joblib.dump(best_model, "salary_prediction_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [12]:
loaded_model = joblib.load("salary_prediction_model.pkl")

print(loaded_model)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=3, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=-1, num_parallel_tree=None, ...)


In [13]:
sample = X_test.iloc[:5]

original_prediction = best_model.predict(sample)

loaded_prediction = loaded_model.predict(sample)

print("Original model predictions:")
print(original_prediction)

print("\nLoaded model predictions:")
print(loaded_prediction)

Original model predictions:
[37820.266 85441.67  42884.207 52073.    36144.168]

Loaded model predictions:
[37820.266 85441.67  42884.207 52073.    36144.168]
